<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week3/Week3_LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Topics: Binary Classification, logistic regression.

In classification our target variable no longer takes on continuous values but instead discrete labels.  In the simplest case (binary classification), we have $y\in \{0,1\}$.  This change of the data means that instead of predicting the target directly our model instead must predict the label probabilities, in the binary case we need only have the model predict $P(y=1|x)$.

For a more explicit example, our target will be the species of iris flower ($0$ denoting the species is setosa, $1$ denoting the species is versicolor) and our features will be measurements of their botanical parts (sepal length, sepal width, petal length, petal width).  So our model is some function of the botanical part measurements that predicts the probability of the species being versicolor.  

We could technically use linear regression to model the probability, but this is typically inadvisable as the outputs of linear regression are not bound within $[0,1]$ and we wouldn't expect the probability to scale linearly with the features.

What we instead will do is take a linear model of the features and plug it into a non-linear function that is bound within $[0,1]$.  So we take $z=\beta_0+\beta_1x_1+...+\beta_px_p$ and plug it into the sigmoid function (also called the logistic function): $\sigma(z)=\frac{1}{1+e^{-z}}$.  Graphically the sigmoid function makes an S-shaped curve with asymptotes at $0$ and $1$.  This model is called logistic regression.

To summarize the logistic regression model is $p(x) = P(y=1|x)=\frac{1}{1+e^{-(\beta_0+\beta_1x_1+...+\beta_px_p)}}$.

We can interpret the coefficients in terms of log-odds:

$$\text{odds: }\frac{p(x)}{1-p(x)}=\frac{\frac{1}{1+e^{-z}}}{1-\frac{1}{1+e^{-z}}}=\frac{\frac{1}{1+e^{-z}}}{\frac{e^{-z}}{1+e^{-z}}}=\frac{1}{e^{-z}}=e^z$$

$$\text{log-odds: } \ln(e^z)=z=\beta_0+\beta_1x_1+...+\beta_px_p$$

So if $x_1$ goes up by 1 unit, the log-odds increase by $\beta_1$.  Equivalently the odds get multiplied by $e^{\beta_1}$.

Now our loss function also must change as we are no longer predicting continuous values for $y$ but instead class probabilities.  Note also that we do not really "observe" any class probabilities in our data, only the actual classes, meaning the "observed" probabilities are effectively $0$ or $1$.

Here it is useful to think in terms of maximum likelihood (which we convert to minimizing the negative likelihood):

$$\text{Let }p_i=P(y_i=1|x_i).$$

$$\text{So }P(y_i=0|x_i)=1-p_i.$$

$$\text{So } y_i\sim\text{Bernoulli}(p_i).$$

Now we can write the likelihood:

$$L(\beta)=\prod_{i=1}^np^{y_i}(1-p_i)^{1-y_i}$$

The log-likelihood is then:

$$\ell(\beta)=\sum_{i=1}^n[y_i\ln(p_i)+(1-y_i)\ln(1-p_i)]$$

Hopefully it is clear that when $y_i=1$ having $p_i$ close to $1$ makes the likelihood larger as does having $p_i$ close to $0$ when $y_i=0$.  

Now, since we fit models by minimizing loss, but in this case we want to maximize the likelihood, we just set our loss to be the negative log-likelihood:

$$\mathcal{L}=-\sum_{i=1}^n[y_i\ln(p_i)+(1-y_i)\ln(1-p_i)]$$

This is referred to ass cross-entropy loss and is the standard loss function used for classification (it also extends naturally to classification problems with more than two classes).  Also note that since $p_i$ is obtained via the logistic regression model, the loss function is still a function of $\beta$.

Brief tangent: using MSE loss for linear regression is also aligned with maximum likelihood if we assume that the random errors have a Normal distribution with a constant variance.

We do not have a closed-form solution here, so we have to use gradient descent or other iterative methods to solve for the parameters that mininmize the loss function.






In [24]:
#imports
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import csv
from sklearn.datasets import load_iris
# set seed
torch.manual_seed(801)

# load data
iris = load_iris()
X = iris['data']
y = iris['target']

# remove observations for third species class to make binary target
X = X[y<2]
y = y[y<2]

# convert to tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)


# split and standardize
def train_test_split(X, y, test_pct=0.2, seed=801):
    torch.manual_seed(seed)
    n = X.shape[0]
    perm = torch.randperm(n)

    split = int(n * (1 - test_pct))
    train_idx = perm[:split]
    test_idx = perm[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(dim=0, keepdim=True)
    std = X_train.std(dim=0, keepdim=True) + 1e-8

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

X_train, X_test, y_train, y_test = train_test_split(X, y)
X_train, X_test = standardize_train_test(X_train, X_test)

# check that we have roughly equal class 0 and class 1 representation in the training data
torch.unique(y_train, return_counts = True)[1]

tensor([41, 39])

In [25]:
'''_, ax = plt.subplots()
scatter = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train)
ax.set(xlabel=iris.feature_names[0], ylabel=iris.feature_names[1])
_ = ax.legend(
    scatter.legend_elements()[0], iris.target_names, loc="upper right", title="Classes"
)'''

'_, ax = plt.subplots()\nscatter = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train)\nax.set(xlabel=iris.feature_names[0], ylabel=iris.feature_names[1])\n_ = ax.legend(\n    scatter.legend_elements()[0], iris.target_names, loc="upper right", title="Classes"\n)'

Set up the model:

In [26]:
p = X_train.shape[1] # number of features (number of columns in X_train)
betas = torch.zeros((p,1), requires_grad = True) # initialize coefficients to 0
beta0 = torch.zeros(1, requires_grad = True) # intercept

# logistic/sigmoid function
def logistic(z):

  return 1/(1 + torch.exp(-z))

# forward pass/probability prediction
def forward(X):
  # returns logistic regression output (predicted P(y=1|x))
  return logistic(X @ betas + beta0)

Define the loss function (binary cross-entropy loss which is the negative log-likelihood):

In [27]:
def cross_entropy(y, p):
  # returns cross entropy loss given true labels (y) and predicted probabilities of being label 1 (p)
  eps = 1e-8 # added for numerical stability so we don't divide by near 0
  return -(y * torch.log(p + eps) + (1 - y) * torch.log(1 - p + eps)).mean() # we are taking the mean here instead of the sum, but it is proportional to the sum so we should get the same results

Set up gradient descent training:

In [28]:
optimizer = torch.optim.SGD([betas, beta0], lr=0.1) # gradient descent optimizer, parameters are the beta coefficients and beta0 intercept, learning rate of 0.1

num_epochs = 3000

# training loop
for epoch in range(num_epochs):
  optimizer.zero_grad() # zero the gradients before proceeding with each iteration

  # forward pass
  p = forward(X_train) # predicted probability
  loss = cross_entropy(y_train, p) # cross-entropy loss

  # backward pass
  loss.backward() # compute gradients
  optimizer.step() # update parameters

  # print loss every 250 epochs
  if ((epoch + 1) % 250 == 0) or (epoch == 0):
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.6931
Epoch 250, Loss: 0.0197
Epoch 500, Loss: 0.0103
Epoch 750, Loss: 0.0070
Epoch 1000, Loss: 0.0054
Epoch 1250, Loss: 0.0043
Epoch 1500, Loss: 0.0037
Epoch 1750, Loss: 0.0032
Epoch 2000, Loss: 0.0028
Epoch 2250, Loss: 0.0025
Epoch 2500, Loss: 0.0023
Epoch 2750, Loss: 0.0021
Epoch 3000, Loss: 0.0019


Now we have fit the model, but note that it only predicts the probability of the iris being versicolor (class 1), so we must use that probability to get an actual label prediction.  The most straightforward cutoff to choose in the binary case is to predict label 1 if the predicted probability for label 1 is at least $0.5$.

In [29]:
def predict(X):
  with torch.no_grad(): # disable gradients when predicting
    p = forward(X) # get predicted probability
    bool_ = p >= 0.5 # true if predicted probability at least 0.5
    return bool_.float() # predict label 1/class 1 if probability of label 1 is at least 0.5

We can also check our accuracy on the train and test sets when using this cutoff:

In [30]:
def accuracy(y_true, y_pred):
  # returns accuracy given true labels (y_true) and predicted labels (y_pred)
    return (y_true == y_pred).float().mean().item()

train_acc = accuracy(y_train, predict(X_train))
test_acc = accuracy(y_test, predict(X_test))

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 1.0
Test Accuracy: 1.0


Conceptual Questions:

1. Why do we have to plug the linear model output into a sigmoid or similar function?

2. Why do we use cross-entropy loss instead of MSE loss?  (Think about 160C).

3. (Review Question) Why do we still standardize the features even though we are not using any penalty/weight decay here?

4. What is the predicted probability $P(y_i=1|x_i)$ when $\beta_0+\beta_1x_{i1}+...+\beta_px_{ip}=0$.

5. What happens if we increase just $\beta_0$ in terms of predicted probabilities?

6.  Assume we predict $y_i$ to have label $1$ if $P(y_i=1|x_i)\geq 0.5$.  What happens to $\hat y_i$ if we multiply all the weights ($0$ through $p$) by $10$?  (Think about Question 4 and what happens to $\beta_0+\beta_1x_{i1}+...+\beta_px_{ip}$ and then what happens to $P(y_i=1|x_i)$).

7.  If I fit model A on the training data and then fit model B on the same training data except I've swapped labels (label 1 becomes label 0 and vice-versa), how do the weights between model A and model B compare?

8.  When might high accuracy be misleading? (Think about the data).

9.  Can two models have the same train/test accuracy but be different?

10.  Why is accuracy not the best metric to use to fit the model?

11.  Can the loss change while accuracy stays the same?

12.  Can the loss improve while accuracy gets worse (or vice-versa)?

13.  Assume that we add another flower species so we now have 3 possible labels, IE $y\in\{1,2,3\}$ now and we have a model that predicts $P(y_i=1|x_i),P(y_i=2|x_i), P(y_i=3|x_i)$.  What is a reasonable way to determine $\hat y_i$ in this setup?

